In [27]:
import pandas as pd
! uv pip install huggingface_hub
! uv pip install rdkit
! uv pip install scikit_learn
import sys
sys.path.append("openadnet")
from openadnet.src.features_data import fpspecs

Audited 1 package in 86ms
Audited 1 package in 8ms
Audited 1 package in 8ms


In [28]:
import pandas as pd

train         = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TRAIN.csv")
test          = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_TEST_BLINDED.csv")
train_counter  = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_counter-assay_TRAIN.csv")
test_structure = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_structure_TEST_BLINDED.csv")
train_single   = pd.read_csv("hf://datasets/openadmet/pxr-challenge-train-test/pxr-challenge_single_concentration_TRAIN.csv")

In [29]:
from rdkit import Chem
import pandas as pd

from rdkit.Chem import PandasTools

# Convert SMILES to RDKit Mol objects
rdkit_mols = list(train["SMILES"].apply(Chem.MolFromSmiles))
PandasTools.AddMoleculeColumnToFrame(train, smilesCol="SMILES", molCol="mol")

In [32]:
train.keys()

Index(['Molecule Name', 'SMILES', 'OCNT Batch', 'pEC50',
       'pEC50_ci.lower (-log10(molarity))',
       'pEC50_ci.upper (-log10(molarity))',
       'Emax_estimate (log2FC vs. baseline)',
       'Emax_ci.lower (log2FC vs. baseline)',
       'Emax_ci.upper (log2FC vs. baseline)',
       'Emax.vs.pos.ctrl_estimate (dimensionless)',
       'Emax.vs.pos.ctrl_ci.lower (dimensionless)',
       'Emax.vs.pos.ctrl_ci.upper (dimensionless)',
       'pEC50_std.error (-log10(molarity))',
       'Emax_std.error (log2FC vs. baseline)',
       'Emax.vs.pos.ctrl_std.error (dimensionless)', 'Split', 'mol'],
      dtype='object')

In [30]:
train.describe()

,pEC50,pEC50_ci.lower (-log10(molarity)),pEC50_ci.upper (-log10(molarity)),Emax_estimate (log2FC vs. baseline),Emax_ci.lower (log2FC vs. baseline),Emax_ci.upper (log2FC vs. baseline),Emax.vs.pos.ctrl_estimate (dimensionless),Emax.vs.pos.ctrl_ci.lower (dimensionless),Emax.vs.pos.ctrl_ci.upper (dimensionless),pEC50_std.error (-log10(molarity)),Emax_std.error (log2FC vs. baseline),Emax.vs.pos.ctrl_std.error (dimensionless)
count,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000,4140.000000
mean,4.320814,3.931155,4.654818,2.600377,2.200883,3.122370,1.074789,0.748300,1.654298,0.191781,0.239551,0.237104
std,1.121600,1.351114,0.887092,0.639018,0.605790,0.753934,0.517648,0.400206,0.828740,0.150968,0.156073,0.175559
min,1.610000,1.020000,2.590000,1.590000,1.470000,1.600000,0.205000,0.177000,0.222000,0.003254,0.002284,0.003126
25%,3.640000,3.207500,4.030000,2.150000,1.660000,2.610000,0.783000,0.511000,1.040000,0.080775,0.096475,0.078750
50%,4.650000,4.310000,4.910000,2.470000,2.070000,3.010000,0.998000,0.668000,1.560000,0.150000,0.202000,0.190000
75%,5.130000,4.930000,5.290000,3.030000,2.530000,3.600000,1.220000,0.843000,2.150000,0.243000,0.376000,0.394000
max,7.548554,7.491205,7.605422,5.740000,5.420000,6.270000,12.500000,8.150000,17.800000,0.739000,0.677000,2.440000


In [7]:
fpspecs

{'morgan_bits_2048_r2': <rdkit.Chem.rdFingerprintGenerator.FingerprintGenerator64 at 0x110068200>,
 'rdkit_bits_2048': <rdkit.Chem.rdFingerprintGenerator.FingerprintGenerator64 at 0x110068270>,
 'atom_pair_sparse': <rdkit.Chem.rdFingerprintGenerator.FingerprintGenerator64 at 0x110068350>,
 'torsion_sparse': <rdkit.Chem.rdFingerprintGenerator.FingerprintGenerator64 at 0x1100683c0>}

In [18]:
fps = fpspecs["morgan_bits_2048_r2"].GetCountFingerprints(rdkit_mols, numThreads=1)
import numpy as np


array(<rdkit.DataStructs.cDataStructs.UIntSparseIntVect object at 0x11346a180>,
      dtype=object)

In [26]:
gen?

Type:           FingerprintGenerator64
String form:    <rdkit.Chem.rdFingerprintGenerator.FingerprintGenerator64 object at 0x1100683c0>
File:           ~/.cache/uv/archive-v0/xFSWzXDdUD9UJ-VA8lD2k/lib/python3.13/site-packages/rdkit/Chem/rdFingerprintGenerator.so
Docstring:      <no docstring>
Init docstring:
Raises an exception
This class cannot be instantiated from Python

In [20]:
from openadnet.src.features_data import fpspecs

fp_table = []
for name, gen in fpspecs.items():
    fps = gen.GetFingerprints(rdkit_mols, numThreads=1)
    for i, fp in enumerate(fps):
        fp_table.append({"mol_idx": i, "fp_name": name, "fp": fp})

fp_df = pd.DataFrame(fp_table)

In [25]:
fp_df["fp"]

0        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                               ...                        
16555    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
16556    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
16557    [0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
16558    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
16559    [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: fp, Length: 16560, dtype: object

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, BaggingRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import numpy as np
y = train["pEC50"] < train["pEC50"].mean()

results = []
for name, gen in fpspecs.items():
    X = np.array([gen.GetFingerprintAsNumPy(m) for m in rdkit_mols])
    model = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("clf", RandomForestRegressor(n_estimators=300, random_state=0)),
    ])
    score = cross_val_score(model, X, y, cv=3, scoring="roc_auc").mean()
    results.append((name, score))

results = sorted(results, key=lambda x: x[1], reverse=True)
print(results)

/Users/ericboittier/.cache/uv/archive-v0/xFSWzXDdUD9UJ-VA8lD2k/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:945: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/ericboittier/.cache/uv/archive-v0/xFSWzXDdUD9UJ-VA8lD2k/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 166, in __call__
    score = scorer._score(
        cached_call, estimator, *args, **routed_params.get(name).score
    )
  File "/Users/ericboittier/.cache/uv/archive-v0/xFSWzXDdUD9UJ-VA8lD2k/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 408, in _score
    response_method = _check_response_method(estimator, self._response_method)
  File "/Users/ericboittier/.cache/uv/archive-v0/xFSWzXDdUD9UJ-VA8lD2k/lib/python3.13/site-packages/sklearn/utils/validation.py", line 2261, in _check_response_method
    raise AttributeError(
    ...<2 lines>...
    )
Attribut

In [ ]:
cross_val_score?

In [ ]:
from sklearn.model_selection import cross_val_score, cross_val_predict

cross_val_predict?